## 🔗 Follow Me

<p align="center">
  <a href="https://github.com/Ramakm">
    <img src="https://img.shields.io/badge/GitHub-Follow-black?style=for-the-badge&logo=github" />
  </a>
  &nbsp;
  <a href="https://www.instagram.com/techwith.ram/">
    <img src="https://img.shields.io/badge/Instagram-Follow-E4405F?style=for-the-badge&logo=instagram&logoColor=white" />
  </a>
  &nbsp;
  <a href="https://x.com/techwith_ram">
    <img src="https://img.shields.io/badge/Twitter-Follow-1DA1F2?style=for-the-badge&logo=twitter&logoColor=white" />
  </a>
</p>

# 01 · Why fine-tune an LLM?

Project: **fine-tune `Llama-3.2-3B-Instruct`** into a domain assistant that follows our instructions and answer style, using **QLoRA** (4-bit base + small trainable adapters) so it fits on a free 16 GB T4.

In this notebook (CPU-only):
1. What a pretrained model is good / bad at
2. When fine-tuning is the right tool
3. The *math* of why LoRA makes it cheap for Llama-3.2-3B

## 1. What fine-tuning changes

| Pretrained / instruct model | After fine-tuning |
|---|---|
| Generic knowledge, generic tone | Your tone, format, vocabulary |
| Needs long prompts with examples (few-shot) | Behaviour is baked into the weights; short prompts |
| Inconsistent output format | Reliable structure (JSON, templates, style) |
| Big model needed for quality | Small model (3B) can match a big one on a *narrow* task |

**Fine-tuning teaches behaviour/style/format. It is a poor way to add lots of new facts** (use RAG for that, see notebook 02).

### Good reasons to fine-tune
- Consistent output format or persona
- Lower latency / cost: a 3B model on your own hardware instead of a large API model
- Privacy: data never leaves your machine
- Long, repetitive prompts you want to shorten

### Bad reasons
- "The model doesn't know my docs" → RAG
- You have < ~200 quality examples and haven't tried prompting
- Facts change weekly

## 2. Why LoRA/QLoRA makes it affordable

Llama-3.2-3B architecture numbers (from its `config.json`):

In [ ]:
# CPU-only: parameter math for Llama-3.2-3B
hidden, layers, inter = 3072, 28, 8192
heads, kv_heads, head_dim = 24, 8, 128
vocab = 128256

q_out  = heads * head_dim      # 3072
kv_out = kv_heads * head_dim   # 1024

# (in_features, out_features) of every linear layer in one transformer block
linears = {
    "q_proj": (hidden, q_out), "k_proj": (hidden, kv_out), "v_proj": (hidden, kv_out), "o_proj": (q_out, hidden),
    "gate_proj": (hidden, inter), "up_proj": (hidden, inter), "down_proj": (inter, hidden),
}

per_layer = sum(i*o for i, o in linears.values())
total = layers*per_layer + vocab*hidden            # embeddings are tied with lm_head in 3.2
print(f"≈ total params: {total/1e9:.2f} B")

def lora_params(r):
    # each adapted layer adds A (r x in) and B (out x r)
    return layers * sum(r*(i+o) for i, o in linears.values())

for r in (8, 16, 32, 64):
    p = lora_params(r)
    print(f"LoRA r={r:>2}: {p/1e6:6.1f} M trainable  ({100*p/total:.2f}% of the model)")

**Takeaway:** with `r=16` on all linear layers we train under **1 %** of the weights (~24 M of 3.2 B). Optimizer states (Adam keeps 2 extra copies) are only needed for those, which is what makes a 3B fine-tune fit on one small GPU.

Next → `02_Where_finetuning_fits_in.ipynb`